In [2]:
import os
import re
import json
import shutil
from pathlib import Path

from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from groq import Groq

load_dotenv()

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_21284\3967941098.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader
d:\Project\Amlgo\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [3]:
load_dotenv()

PDF_PATH = os.getenv("DATA_PATH")
CHROMA_DIR = os.getenv("CHROMA_DIR")
EMBED_MODEL = os.getenv("EMBED_MODEL")
GROQ_MODEL = os.getenv("GROQ_MODEL")

In [5]:
shutil.rmtree("./vectordb", ignore_errors=True)

In [6]:
loader = PyMuPDFLoader(PDF_PATH)

documents = loader.load()

print(f"\nLoaded {len(documents)} pages\n")

for i, doc in enumerate(documents, start=1):
    print(f"Page {i}: {len(doc.page_content)} chars")


Loaded 20 pages

Page 1: 3637 chars
Page 2: 2692 chars
Page 3: 3521 chars
Page 4: 3728 chars
Page 5: 3280 chars
Page 6: 3108 chars
Page 7: 3756 chars
Page 8: 3892 chars
Page 9: 3438 chars
Page 10: 2907 chars
Page 11: 3787 chars
Page 12: 2999 chars
Page 13: 3166 chars
Page 14: 3969 chars
Page 15: 4009 chars
Page 16: 3996 chars
Page 17: 3985 chars
Page 18: 3971 chars
Page 19: 3550 chars
Page 20: 871 chars


In [7]:
documents[0].page_content

'User Agreement \n1. Introduction \nThis User Agreement, the Mobile Application Terms of Use, and all policies and additional terms \nposted on and in our sites, applications, tools, and services (collectively "Services") set out the terms \non which eBay offers you access to and use of our Services. You can find an overview of our policies \nhere. The Mobile Application Terms of Use, all policies, and additional terms posted on and in our \nServices are incorporated into this User Agreement. You agree to comply with all terms of this User \nAgreement when accessing or using our Services. \nThe entity you are contracting with is: eBay Inc., 2025 Hamilton Ave., San Jose, CA 95125, if you \nreside in the United States; eBay (UK) Limited, 1 More London Place, London, SE1 2AF, United \nKingdom, if you reside in the United Kingdom; eBay GmbH, Albert-Einstein-Ring 2-6, 14532 \nKleinmachnow, Germany, if you reside in the European Union; eBay Canada Limited, 240 Richmond \nStreet West, 2nd Flo

In [8]:
def clean(text: str) -> str:
    text = re.sub(r'\s+',   ' ',  text)   # collapse whitespace / newlines
    text = re.sub(r'\.{3,}','.', text)   # remove ellipsis noise
    text = re.sub(r'-{2,}', '-',  text)   # normalise dashes
    return text.strip()

# print(f'Before : {len(raw_text):,} chars')
# print(f'After  : {len(clean_doc):,} chars')
# print('\n--- Sample ---')


In [9]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200
)

In [10]:
chunk_docs = []

for page_num, page in enumerate(documents, start=1):

    page_text = clean(page.page_content)

    chunks = splitter.split_text(page_text)

    for chunk_id, chunk in enumerate(chunks):

        chunk_docs.append(
            Document(
                page_content=chunk,
                metadata={
                    "page": page_num,
                    "chunk": chunk_id
                }
            )
        )

print(f"Created {len(chunk_docs)} chunks")

Created 72 chunks


In [11]:
Path("chunks").mkdir(exist_ok=True)

with open("chunks/chunks.json", "w", encoding="utf-8") as f:

    json.dump(
        [
            {
                "id": i,
                "page": d.metadata["page"],
                "text": d.page_content
            }
            for i, d in enumerate(chunk_docs)
        ],
        f,
        indent=2,
        ensure_ascii=False
    )

print("Saved chunks/chunks.json")

Saved chunks/chunks.json


In [12]:
embedding_model = HuggingFaceEmbeddings(
    model_name=os.getenv("EMBED_MODEL")
)

vector_store = Chroma.from_documents(
    documents=chunk_docs,
    embedding=embedding_model,
    collection_name="rag_docs",
    persist_directory=os.getenv("CHROMA_DIR")
)

print(
    f"Stored {vector_store._collection.count()} vectors"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6494.15it/s]


Stored 72 vectors


In [ ]:
# ── On subsequent runs: load existing store instead of rebuilding ──
vector_store = Chroma(
    collection_name='rag_docs',
    embedding_function=embedding_model,
    persist_directory=os.getenv("CHROMA_DIR"),
)
print(f'Loaded store — {vector_store._collection.count()} vectors')

✓ Loaded store — 72 vectors


In [ ]:
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 6,
        "fetch_k": 20
    }
)
query = 'What happens when a buyer purchases an item on eBay?'
docs  = retriever.invoke(query)

print(f'Retrieved {len(docs)} chunks for: "{query}"\n')
for i, doc in enumerate(docs, 1):
    print(f'Chunk {i}')
    print(doc.page_content[:300])
    print()

Retrieved 6 chunks for: "What happens when a buyer purchases an item on eBay?"

── Chunk 1 ──
or if a transaction is canceled after payment has been completed, eBay may issue a refund to the buyer on the seller's behalf and charge the seller for the amount of the refund. Additionally, eBay may charge sellers for the cost of return shipping labels and/or other reasonable fees from sellers whe

── Chunk 2 ──
including across any eBay Services, our partners, or third-party property or advertising medium; and • Selling fees do not purchase exclusive rights to item exposure on our Services. We may display third-party advertisements (including links and references thereto) or other content in any part of ou

── Chunk 3 ──
• number of listings matching the buyer's query. • To drive a positive user experience, a listing may not appear in some search and browse results regardless of the sort order chosen by the buyer; • Some advanced listing upgrades will only be visible on some of our Services

In [106]:
# # Also available: similarity search WITH relevance scores (0–1, higher = better)
# results = vector_store.similarity_search_with_relevance_scores(query, k=4)
# for doc, score in results:
#     print(f'Score {score:.3f} | {doc.page_content[:120]}…')

In [13]:
from groq import Groq
client = Groq(api_key=os.environ['GROQ_API_KEY'])
GROQ_MODEL = os.getenv("GROQ_MODEL")


In [15]:
SYSTEM_PROMPT = '''You are a document question-answering assistant. Answer ONLY from the provided context.
If the answer is present in the context, quote or summarize the relevant text. Do not use outside knowledge.
If the answer is not present in the context, reply exactly: I don't have enough information to answer that.'''

def build_prompt(query, docs):

    context = "\n\n----------------\n\n".join(
        doc.page_content
        for doc in docs
    )
    return [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content":
                f"Context:\n{context}\n\n"
                f"Question: {query}"
        }
    ]


print('✓ Prompt builder ready')

✓ Prompt builder ready


In [16]:
def inspect_retrieval(query, k=6):

    print("\n")
    print("=" * 80)
    print("QUERY")
    print(query)
    print("=" * 80)

    results = vector_store.similarity_search_with_relevance_scores(
        query,
        k=k
    )

    for idx, (doc, score) in enumerate(results, start=1):

        print("\n")
        print("-" * 80)

        print(
            f"Result {idx}"
            f" | Score={score:.4f}"
            f" | Page={doc.metadata['page']}"
        )

        print("-" * 80)

        print(doc.page_content[:700])


In [111]:
# # Non-streaming generation
# messages = build_prompt(query, docs)

# response = client.chat.completions.create(
#     model=GROQ_MODEL,
#     messages=messages,
#     temperature=0.2,
#     max_tokens=512,
#     stream=True,  # ← set to True for streaming responses
# )

# for chunk in response:
#     print(chunk.choices[0].delta.content, end="", flush=True)

In [16]:
def rag(query):

    docs = retriever.invoke(query)

    messages = build_prompt(query, docs)

    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=messages,
        temperature=0.2,
        max_tokens=512
    )

    answer = response.choices[0].message.content

    return {
        "answer": answer,
        "sources": docs
    }

In [17]:
def rag_stream(query):

    docs = retriever.invoke(query)

    messages = build_prompt(query, docs)

    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=messages,
        temperature=0.2,
        max_tokens=512,
        stream=True
    )

    answer = ""

    for chunk in response:

        delta = chunk.choices[0].delta.content

        if delta:
            print(delta, end="", flush=True)
            answer += delta

    print()

    return {
        "answer": answer,
        "sources": docs
    }

In [18]:

query = "What happens when a buyer purchases an item on eBay?"

inspect_retrieval(query)

result = rag_stream(query)

print("\n\nANSWER:\n")
print(result["answer"])

print("\n\nSOURCES:\n")

for source in result["sources"]:

    print(
        f"Page {source.metadata['page']}"
    )



QUERY
What happens when a buyer purchases an item on eBay?


--------------------------------------------------------------------------------
Result 1 | Score=0.6123 | Page=9
--------------------------------------------------------------------------------
or if a transaction is canceled after payment has been completed, eBay may issue a refund to the buyer on the seller's behalf and charge the seller for the amount of the refund. Additionally, eBay may charge sellers for the cost of return shipping labels and/or other reasonable fees from sellers when: • An eBay-generated return shipping label is used, and the seller is responsible for its cost;


--------------------------------------------------------------------------------
Result 2 | Score=0.5987 | Page=6
--------------------------------------------------------------------------------
including across any eBay Services, our partners, or third-party property or advertising medium; and • Selling fees do not purchase exclusive right

In [19]:
query = "Can eBay record phone calls with users? Why?"

inspect_retrieval(query)



QUERY
Can eBay record phone calls with users? Why?


--------------------------------------------------------------------------------
Result 1 | Score=0.6445 | Page=9
--------------------------------------------------------------------------------
retention, and protection of your personal information is governed by our User Privacy Notice. As described in our User Privacy Notice, eBay may collect other telephone numbers for you and may place manual non-marketing calls to any of those numbers and autodialed non-marketing calls to any landline. Standard telephone minute and text charges may apply and may include overage fees if you have exceeded your plan limits. You may change your marketing communications preference for calls at any time, including through the Communication Preferences section of your My eBay. You may also opt out of a specific text marketing campaign by replying "STOP" to such marketing text message. eBay may share 


-----------------------------------------------

In [20]:
for doc, score in vector_store.similarity_search_with_relevance_scores(query, k=20):

    if "purchase conditions" in doc.page_content.lower():

        print("LEN:", len(doc.page_content))
        print(doc.page_content)
        break

LEN: 1067
including across any eBay Services, our partners, or third-party property or advertising medium; and • Selling fees do not purchase exclusive rights to item exposure on our Services. We may display third-party advertisements (including links and references thereto) or other content in any part of our Services, including listings, in our sole discretion and without consent from, or payment, fee reduction, or other credit to, sellers. 8. Purchase Conditions When buying an item using our Services, you agree to the Rules and policies for buyers and that: • You are responsible for reading the full item listing before making a bid or offer, buying, or committing to buy; • You enter into a legally binding contract to purchase an item when you buy the item, commit to buy the item, your offer for the item is accepted, you have the winning bid for the item, or your bid for the item is otherwise accepted, regardless of when payment is due or received unless the transaction terms state t

In [22]:
for doc in chunk_docs:

    if doc.metadata["page"] == 6:

        print("="*80)
        print(doc.metadata)
        print(doc.page_content)

{'page': 6, 'chunk': 0}
• number of listings matching the buyer's query. • To drive a positive user experience, a listing may not appear in some search and browse results regardless of the sort order chosen by the buyer; • Some advanced listing upgrades will only be visible on some of our Services; • eBay's Duplicate listings policy may also affect whether your listing appears in search results; • Metatags and URL links that are included in a listing may be removed or altered; • We may provide you with optional information to consider when creating your listings. Such information may be based on the aggregated sales and performance history of similar sold and/or current listings; results may vary for individual listings; • You agree that we may display the sales and performance history of your individual listings (and their associated campaigns) to others; • For items listed in certain categories, subject to certain programs, and/or offered or sold at certain price points, eBay may req

In [23]:
docs = retriever.invoke(
    "What happens when a buyer purchases an item on eBay?"
)

for d in docs:
    if "legally binding contract" in d.page_content.lower():
        print("FOUND")
        print(d.page_content)

FOUND
including across any eBay Services, our partners, or third-party property or advertising medium; and • Selling fees do not purchase exclusive rights to item exposure on our Services. We may display third-party advertisements (including links and references thereto) or other content in any part of our Services, including listings, in our sole discretion and without consent from, or payment, fee reduction, or other credit to, sellers. 8. Purchase Conditions When buying an item using our Services, you agree to the Rules and policies for buyers and that: • You are responsible for reading the full item listing before making a bid or offer, buying, or committing to buy; • You enter into a legally binding contract to purchase an item when you buy the item, commit to buy the item, your offer for the item is accepted, you have the winning bid for the item, or your bid for the item is otherwise accepted, regardless of when payment is due or received unless the transaction terms state that 